This is the QLoRA fine-tuning process using the jsonl dataset we prepared. We'll use the Unsloth library, which significantly speeds up the training process while reducing memory requirements. This makes it perfectly suited for training on a free Google Colab Tesla T4 GPU.

#Step 1: Setup and Installation

First, we install the unsloth package along with its dependencies. Since we are using a Tesla T4 GPU on Google Colab, we will also need to install xformers (Flash Attention) to optimize memory usage.

In [1]:
%%capture
# 1. Install Unsloth from the stable PyPI release
!pip install unsloth

# 2. Force the installation of specific compatible dependencies for Colab T4
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

#Step 2: Load the Base Model and Tokenizer

We will load the pre-trained SAWithanage/SinLlama-Llama-3-8B-Merged model. Unsloth provides a FastLanguageModel class to handle this efficiently, integrating the 4-bit quantization config.

In [2]:
import torch
from unsloth import FastLanguageModel

# 1. Define configuration
max_seq_length = 2048 # Recommended starting length for testing
dtype = None # Auto-detects (Float16 for Tesla T4)
load_in_4bit = True # Enables 4-bit quantization (QLoRA)

# 2. Load the SinLlama base model
model_id = "SAWithanage/SinLlama-Llama-3-8B-Merged"

print(f"Loading {model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model and Tokenizer loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading SAWithanage/SinLlama-Llama-3-8B-Merged...
==((====))==  Unsloth 2026.8.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load SAWithanage/SinLlama-Llama-3-8B-Merged as a legacy tokenizer.


Unsloth: SAWithanage/SinLlama-Llama-3-8B-Merged has no pad_token. Using pad_token = <|reserved_special_token_250|>.
Model and Tokenizer loaded successfully.


# Step 3: Apply LoRA Adapters
Now, we add the LoRA adapters to the model. This step is crucial because it ensures we are only training a tiny fraction of the 8 billion parameters (typically 1-10%), which is why QLoRA works on a T4 GPU.

In [3]:
print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Controls expressiveness vs memory (suggested: 8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized setting
    bias = "none",    # Optimized setting
    use_gradient_checkpointing = "unsloth", # Crucial for saving VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Adapters configured.")

Applying LoRA adapters...


Unsloth 2026.8.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Adapters configured.


#Step 4: Load and Format and Spilt the Custom JSONL Dataset
We will load the qlora_dataset.jsonl file you generated. Unsloth's standard approach is to map the data into a specific string format before training. We will use the Alpaca template structure.

In [4]:
from datasets import load_dataset

# 1. Load the JSONL dataset
dataset_path = "/content/qlora_dataset.jsonl"
print(f"Loading dataset from {dataset_path}...")
raw_dataset = load_dataset("json", data_files={"train": dataset_path}, split="train")

# 2. Define the Alpaca formatting template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

# 3. Apply formatting function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=instruction, input=input_text, output=output_text) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts }

# 4. Format dataset
formatted_dataset = raw_dataset.map(formatting_prompts_func, batched = True)

# 5. SPLIT DATASET (90% Train / 10% Test)
split_dataset = formatted_dataset.train_test_split(test_size=0.10, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"✅ Total samples: {len(formatted_dataset)}")
print(f"   ├── Training samples: {len(train_dataset)}")
print(f"   └── Evaluation (Testing) samples: {len(eval_dataset)}")

Loading dataset from /content/qlora_dataset.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/126 [00:00<?, ? examples/s]

✅ Total samples: 126
   ├── Training samples: 113
   └── Evaluation (Testing) samples: 13


#Step 5: Training Configuration with SFTTrainer
Finally, we use the SFTTrainer from the trl library to execute the training loop. The parameters below are generally good starting points for small datasets.

In [5]:
# from trl import SFTTrainer, SFTConfig
# from transformers import TrainingArguments

# print("Initializing SFTTrainer with Evaluation split...")
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = train_dataset,
#     eval_dataset = eval_dataset, # Evaluation set added here
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = False,
#     args = SFTConfig(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 5,
#         num_train_epochs = 3,
#         learning_rate = 2e-4,
#         logging_steps = 1,
#         eval_strategy = "steps", # Automatically evaluates performance on test set
#         eval_steps = 5,          # Evaluates every 5 training steps
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         output_dir = "outputs",
#     ),
# )

# # Start training!
# print("Starting Fine-Tuning Process...")
# trainer_stats = trainer.train()
# print("\n--- Fine-Tuning Complete! ---")

Initializing SFTTrainer with Evaluation split...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/113 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/13 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting Fine-Tuning Process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 113 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,162,971,648 (0.51% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5,4.547724,4.152433
10,3.274633,3.539982
15,3.583107,3.373327
20,3.003542,3.246863
25,3.157744,3.167195
30,4.725829,3.110788
35,3.162362,3.069405
40,2.986716,3.042766
45,3.231766,3.032530


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-45/tokenizer_config.json.



--- Fine-Tuning Complete! ---


In [25]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

print("Initializing SFTTrainer with Regularized Hyperparameters...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 3,

        # --- THE OVERFITTING FIXES ---
        num_train_epochs = 1.5,       # Reduced from 3 to 1.5 (Prevents memorization)
        learning_rate = 4e-5,         # Reduced from 2e-4 to 4e-5 (Gentle weight updates)
        weight_decay = 0.1,           # Increased from 0.01 to 0.1 (Regularization penalty)
        # -----------------------------

        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 4,               # Evaluate loss more frequently
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine", # Cosine decay smoothly lowers LR near the end
        seed = 3407,
        output_dir = "outputs_regularized",
    ),
)

# Start Fine-Tuning
print("Starting Regularized Training Run...")
trainer_stats = trainer.train()
print("\n--- Training Complete! ---")

Initializing SFTTrainer with Regularized Hyperparameters...


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/113 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/13 [00:00<?, ? examples/s]

Starting Regularized Training Run...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 113 | Num Epochs = 2 | Total steps = 23
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,162,971,648 (0.51% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
4,2.703060,3.023453
8,3.100867,3.005962
12,2.621047,2.987681
16,2.957149,2.976560
20,2.555419,2.971340
23,2.685392,2.970940


Unsloth: Restored added_tokens_decoder metadata in outputs_regularized/checkpoint-23/tokenizer_config.json.



--- Training Complete! ---


#Test the Fine-Tuned Model

In [32]:
# 1. Switch the model to 2x faster inference mode
FastLanguageModel.for_inference(model)

# 2. Pick a noisy text from your 10% Test Split
test_noisy_text = eval_dataset[5]["input"]

# 3. Format it using the exact Alpaca prompt used during training
test_prompt = alpaca_prompt.format(
    instruction="You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.",
    input=test_noisy_text,
    output="" # Leave output blank for the model to generate
)

# Tokenize and generate
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.1 # Keep it deterministic
)

# Print the result
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("--- MODEL OUTPUT ---")
print(generated_text)

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- MODEL OUTPUT ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.

### Input:
ඕය ම පිනකයි මට පුවන් සයවන්නහෙලෝ න්ටඉන්න හෙලෝ පේඔව් කියන්න සර් හෙලෝ ඔව් මගේ රවුටර බිල් එක මට දැන් මාස දෙක් විතර ගෙවාගන්න බැරි වුනා මේ ම වෙනදට බිල් එක පේ කරන්නේ විදිහට ඒ කරගන්න විදිහන්නේනෑහැ ඔන්ලයින් පමේ පේමන්ට් එකට බිල් වීව් එක කියරලා තමයි යන්නේ ම මට දැන් ඒකේ රවුටර් බිල් එකේ මවුන්ට් එක හරියට දැනගන්නයි කොහොමද මේ ඔන්ලයින් පේමන්ට් එක කරගන්න පුළුවන් විදිහක් පලිය කරගන්න තමයි ගත්තේ හරි ඔය සර්ගේ අදාළ එස් එල්ටී කනෙක්ෂන් එකේ නම්බර් එක කියන්න බංදු දයි හරි යි දෙකයි හරි අයි ර කරුණාකර ඇතුමේ රැඳී ඉන්නඕකරැඳිටාට ස්තූතියි සර් මෙතන කනෙක්ෂන් හිමිකරුගේ නම කියන්න මිස්ටර් විතර් සිංහදඔව් එතකොට සර් කොව වෙනදා කොහොමද පේමන්ට් එක කරන්නේ සර් ලයින් එකට දැන ටරනම් ලයින් එක ස්පේන්

# Evaluating the Model

In [10]:
%%capture
!pip install evaluate jiwer rouge_score sacrebleu

In [30]:
import torch
import evaluate
from tqdm import tqdm

# 1. Load Metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
rouge_metric = evaluate.load("rouge")

FastLanguageModel.for_inference(model)

predictions = []
references = []

print(f"Starting evaluation on {len(eval_dataset)} test samples...")

# 2. Generation Loop with Robust Decoding
for idx in tqdm(range(len(eval_dataset))):
    noisy_input = eval_dataset[idx]["input"]
    true_clean_output = eval_dataset[idx]["output"]

    test_prompt = alpaca_prompt.format(
        instruction="You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.",
        input=noisy_input,
        output=""
    )

    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode with special tokens to catch delimiters
    full_output = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

    # Extract only the response after '### Response:'
    try:
        generated_text = full_output.split("### Response:")[-1]
        # Remove the EOS token and trailing whitespace
        generated_text = generated_text.replace(tokenizer.eos_token, "").strip()
    except Exception:
        generated_text = ""

    predictions.append(generated_text)
    references.append(true_clean_output)

Starting evaluation on 13 test samples...


100%|██████████| 13/13 [04:10<00:00, 19.29s/it]


In [31]:
# 3. Safe Sanity Check (Prevents IndexError)
num_samples = len(references)
print(f"\nCompleted {num_samples} evaluation samples.")

print("\n--- SAMPLE INSPECTION ---")
if num_samples == 0:
    print("❌ No samples were processed. Check if eval_dataset is empty!")
else:
    for i in range(min(3, num_samples)):
        print(f"\n[SAMPLE {i+1}]")
        print(f"EXPECTED : {references[i]}")
        print(f"PREDICTED: {predictions[i]}")
print("--------------------------\n")

# 4. Compute Metrics (Only if predictions exist)
if num_samples > 0:
    wer_score = wer_metric.compute(predictions=predictions, references=references)
    cer_score = cer_metric.compute(predictions=predictions, references=references)
    rouge_score = rouge_metric.compute(predictions=predictions, references=references)

    print("========================================")
    print("        EVALUATION RESULTS              ")
    print("========================================")
    print(f"Word Error Rate (WER):      {wer_score:.4f}  (Lower is better)")
    print(f"Character Error Rate (CER): {cer_score:.4f}  (Lower is better)")
    print("----------------------------------------")
    print(f"ROUGE-1 (Unigram match):    {rouge_score['rouge1']:.4f}  (Higher is better)")
    print(f"ROUGE-2 (Bigram match):     {rouge_score['rouge2']:.4f}  (Higher is better)")
    print(f"ROUGE-L (Sentence flow):    {rouge_score['rougeL']:.4f}  (Higher is better)")
    print("========================================")


Completed 13 evaluation samples.

--- SAMPLE INSPECTION ---

[SAMPLE 1]
EXPECTED : ආයුබෝවන් මම තුෂාරි, මට පුළුවනි ඔබට සහාය වන්න. ආයුබෝවන් මිස්, මගේ මේ නම්බර් එකේ කනෙක්ෂන් එක බ්ලොක් වෙලා තියෙනවා. ඒක පොඩ්ඩක් රීකනෙක්ට් කරගන්න ඕනේ. අද පේමන්ට් එක කරන්න පුළුවන්. රැඳී සිටින්න සර්, චෙක් කරලා බලන්නම්. බිල් එක තියෙනවා, පේමන්ට් එක කරන්න ඕනේ සර්. සාමාන්‍යයෙන් කීයක් වගේද පේමන්ට් එක දාන්න තියෙන්නේ? ඉන්ටර්නෙට් කනෙක්ට් වෙනවා, මොකද ස්ලෝ කරලා තියෙන්නේ. ඉන්ටර්නෙට් යන්න පුළුවන්, ස්ලෝ කරලා තියෙන්නේ. ආ ඕකේ මම දැන් පේමන්ට් එක දාන්නම්. තැන්ක් යු. වෙනත් යමක් දැනගන්න අවශ්‍යද? නැහැ මිස්. මා ලබා දුන් සේවය ඇගයීම සඳහා රැඳී සිටින්න. එස් එල් ටී මොබිටෙල් ඇමතුවට ස්තූතියි. සුභ දවසක්.
PREDICTED: ආයුබෝවන්, මම තුෂාරි. මට පුළුවනි ඔබට සහාය වන්න. ආයුබෝවන් මිස්, මගේ නම්බර් එක එක කනෙක්ෂන් එක වෙලා තියෙනවා. ඒක රීකනෙක්ට් කරගන්න ඕනේ. මම අද පේමන්ට් එක කරන්න පුළුවන්. නම තියෙන්නේ කොහොමද? එම්.ආර්.ටී. රැඳී ඉන්න. චෙක් කරලා බලන්නම්. ඇමතුමේ රැඳී සිටින්න. බිල් එක තියෙනවා. පේමන්ට් එක කරන්න ඕනේ සර්. ඇමතුමේ රැඳී සිටින්න. කනෙක්ට් වෙන්නේ නෑ සර්

In [33]:
# 1. Ensure inference mode
FastLanguageModel.for_inference(model)

# 2. Grab the first test sample
noisy_input = eval_dataset[0]["input"]
expected_output = eval_dataset[0]["output"]

# 3. Format prompt
test_prompt = alpaca_prompt.format(
    instruction="You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.",
    input=noisy_input,
    output=""
)

# 4. Generate
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Add specific Llama-3 stopping criteria
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.1,
    eos_token_id=terminators,
    pad_token_id=tokenizer.eos_token_id
)

# 5. Decode WITH special tokens visible
raw_output = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

print("=== RAW OUTPUT FROM TOKENIZER ===")
print(repr(raw_output)) # repr() shows all hidden \n and spaces
print("\n=== EXPECTED OUTPUT ===")
print(repr(expected_output))

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RAW OUTPUT FROM TOKENIZER ===
'<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nYou are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.\n\n### Input:\nආයුබෝවන් ම තුෂාරි මට පුළුවනි ඔබට සහාය වන්න ආයුබෝවන් මිස් මගේ මේ නම්බර් එක එක කනෙක්ෂන් එක බව වෙලා තියෙනවා හ්ම් ඒක පොඩ්නල් රීකනෙක්ට් කරගන්න ඕනේ ම අද වසට පේමන්ට් එක කරන්න පුළුවන් නම තියෙන්නේ කොහොමද එම්අාර් රැඳී ක් චෙක් කරලා බලන්නම් ඇමතුමේ රැඳී ිටින්න හරි ඇමතුමේ රැඳී සිටින්න සර් ඕකේ බිල් එක තියෙනවා යි සත් ඇය ඇතුළ පේමන්ට් එක කරන්න ඕනේ සර් ඇමතුමේ රැ සිටින්න හරි හරි කනෙක්ට් වෙන්නේ නෑ සර් පේමන්ට් එක කරන්න ඕනේ ආ සාමාන්යෙන් කීයක් වගේද පේමන්ට් එක දාන්න තියෙන්නේ අ ඉන්ටරනෙට් නම් කනෙක්ට් වුනාවගේ ක් ගෙ වෙන්න ඕනේ සර් ා ඉන්ට්නෙට් කනෙක්ට් වෙනවා මොකද ස්ලෝ කරලා තියෙන්නේ රයිට් ඉන්ටර්නෙට්ට් යන්න බැරි කමක් නැ සර්ට ඉන්ටරනෙට් යන්න පුළු

#Merge and Export to GGUF

In [34]:
# 1. Free up VRAM before exporting
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

# 2. Export to GGUF Format
print("Merging LoRA adapters and converting to GGUF...")

model.save_pretrained_gguf(
    "SinLlama_ASR_Cleaner_GGUF",
    tokenizer,
    quantization_method = "q4_k_m" # The optimal 4-bit compression for local CPU inference
)

print("\n✅ Export Complete! You can now download the .gguf file from the Colab file browser.")

Merging LoRA adapters and converting to GGUF...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in SinLlama_ASR_Cleaner_GGUF/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...



Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:   0%|          | 0/9 [00:00<?, ?it/s]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  11%|█         | 1/9 [00:18<02:27, 18.41s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  22%|██▏       | 2/9 [01:04<04:03, 34.84s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  33%|███▎      | 3/9 [01:50<03:57, 39.61s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  44%|████▍     | 4/9 [02:38<03:36, 43.20s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  56%|█████▌    | 5/9 [03:24<02:56, 44.02s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  67%|██████▋   | 6/9 [04:11<02:15, 45.26s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  78%|███████▊  | 7/9 [05:00<01:33, 46.52s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`:  89%|████████▉ | 8

Successfully copied all 9 files from cache to `SinLlama_ASR_Cleaner_GGUF`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 9/9 [00:00<00:00, 64638.25it/s]


Unsloth: Merging weights into 16bit:   0%|          | 0/9 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  11%|█         | 1/9 [00:21<02:54, 21.85s/it]

Unsloth: Merging weights into 16bit:  22%|██▏       | 2/9 [01:06<04:07, 35.34s/it]

Unsloth: Merging weights into 16bit:  33%|███▎      | 3/9 [02:04<04:35, 45.84s/it]

Unsloth: Merging weights into 16bit:  44%|████▍     | 4/9 [03:00<04:08, 49.79s/it]

Unsloth: Merging weights into 16bit:  56%|█████▌    | 5/9 [03:59<03:32, 53.12s/it]

Unsloth: Merging weights into 16bit:  67%|██████▋   | 6/9 [04:54<02:41, 53.77s/it]

Unsloth: Merging weights into 16bit:  78%|███████▊  | 7/9 [05:56<01:52, 56.32s/it]

Unsloth: Merging weights into 16bit:  89%|████████▉ | 8/9 [06:49<00:55, 55.43s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 9/9 [07:31<00:00, 50.16s/it]


Unsloth: Merge process complete. Saved to `/content/SinLlama_ASR_Cleaner_GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10225-mix-345e1e3 (app-b10225-mix-345e1e3-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['SinLlama_ASR_Cleaner_GGUF_gguf/SinLlama-Llama-3-8B-Merged.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


KeyboardInterrupt: 

In [35]:
# This saves ONLY your trained LoRA adapters (a tiny ~150MB file)
model.save_pretrained("lora_backup")
tokenizer.save_pretrained("lora_backup")

print("Backup saved! You can download the 'lora_backup' folder.")

Unsloth: Restored added_tokens_decoder metadata in lora_backup/tokenizer_config.json.


Backup saved! You can download the 'lora_backup' folder.
